# El recocido simulado como cadena de Markov: una aplicación al problema del agente viajero

**Proyecto final — Cadenas de Markov, Universidad Nacional de Colombia**

Henry Giovani Barrera Martínez · Laura Victoria Torres Pinilla · TODO: tercer integrante

Este notebook contiene el código completo de las simulaciones del informe. Cada sección
corresponde a un experimento de la sección *Resultados de las simulaciones y análisis*:

| Sección | Qué muestra | Resultado teórico que verifica |
|---|---|---|
| 1–3 | Instancias, rutas, cadena de Metropolis, recocido y QUBO | — |
| 4 | Verificaciones de la cadena con $n=5$ | Proposiciones de vecindad, irreducibilidad, balance detallado y espectro |
| 5 | Distribución de Boltzmann según la temperatura | Proposición del papel de la temperatura |
| 6 | Frecuencias empíricas frente a la distribución exacta | Teorema ergódico |
| 7 | Brecha espectral y profundidad crítica | Teorema de la brecha a temperatura baja |
| 8 | Recocido simulado en instancias de TSPLIB | Recocido con enfriamiento geométrico |
| 9 | Recocido sobre el QUBO y sensibilidad a $M$ | Proposición de la barrera de penalización |

**Reproducibilidad.** Cada experimento fija su semilla con `numpy.random.default_rng`, de
modo que al ejecutar *Entorno de ejecución → Ejecutar todas* se obtienen exactamente las
mismas tablas y figuras. Las instancias de TSPLIB están incrustadas como coordenadas, así
que no se descarga nada. La ejecución completa tarda unos minutos.

Las figuras y tablas se guardan en la carpeta `results/` (en PDF para el informe y en PNG).

## 0. Configuración

In [ ]:
import itertools
import math
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Carpeta de salida: ../results si se ejecuta desde notebook/ en el repositorio,
# results/ en cualquier otro caso (por ejemplo, en Colab).
RESULTADOS = Path("../results") if Path("../results").is_dir() else Path("results")
RESULTADOS.mkdir(exist_ok=True)

# Paleta categórica en orden fijo y tintas para texto.
AZUL, NARANJA, AGUAMARINA, AMARILLO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
TINTA, TINTA_2 = "#0b0b0b", "#52514e"

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "font.size": 10,
    "axes.edgecolor": TINTA_2,
    "axes.labelcolor": TINTA,
    "axes.titlesize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "grid.linewidth": 0.6,
    "xtick.color": TINTA_2,
    "ytick.color": TINTA_2,
    "lines.linewidth": 2,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def guardar(fig, nombre):
    # Guarda la figura en PDF (para LaTeX) y PNG.
    for ext in ("pdf", "png"):
        fig.savefig(RESULTADOS / f"{nombre}.{ext}", bbox_inches="tight")
    print(f"figura guardada: {RESULTADOS / nombre}.pdf")


print("Resultados en:", RESULTADOS.resolve())

## 1. Instancias del TSP

Se usan dos tipos de instancias:

* **Aleatorias**: $n$ puntos uniformes en $[0,1000]^2$, con distancia euclidiana redondeada
  al entero más cercano (convención `EUC_2D` de TSPLIB).
* **TSPLIB** (Reinelt, 1991): `burma14`, `ulysses16`, `ulysses22` (distancia geográfica
  `GEO`) y `berlin52` (`EUC_2D`), cuyos óptimos están publicados.

In [ ]:
# Coordenadas de TSPLIB, copiadas de los archivos .tsp originales.
TSPLIB = {
    "burma14": ("GEO", [
        (16.47, 96.10), (16.47, 94.44), (20.09, 92.54), (22.39, 93.37), (25.23, 97.24),
        (22.00, 96.05), (20.47, 97.02), (17.20, 96.29), (16.30, 97.38), (14.05, 98.12),
        (16.53, 97.38), (21.52, 95.59), (19.41, 97.13), (20.09, 94.55)]),
    "ulysses16": ("GEO", [
        (38.24, 20.42), (39.57, 26.15), (40.56, 25.32), (36.26, 23.12), (33.48, 10.54),
        (37.56, 12.19), (38.42, 13.11), (37.52, 20.44), (41.23, 9.10), (41.17, 13.05),
        (36.08, -5.21), (38.47, 15.13), (38.15, 15.35), (37.51, 15.17), (35.49, 14.32),
        (39.36, 19.56)]),
    "ulysses22": ("GEO", [
        (38.24, 20.42), (39.57, 26.15), (40.56, 25.32), (36.26, 23.12), (33.48, 10.54),
        (37.56, 12.19), (38.42, 13.11), (37.52, 20.44), (41.23, 9.10), (41.17, 13.05),
        (36.08, -5.21), (38.47, 15.13), (38.15, 15.35), (37.51, 15.17), (35.49, 14.32),
        (39.36, 19.56), (38.09, 24.36), (36.09, 23.00), (40.44, 13.57), (40.33, 14.15),
        (40.37, 14.23), (37.57, 22.56)]),
    "berlin52": ("EUC_2D", [
        (565, 575), (25, 185), (345, 750), (945, 685), (845, 655), (880, 660), (25, 230),
        (525, 1000), (580, 1175), (650, 1130), (1605, 620), (1220, 580), (1465, 200),
        (1530, 5), (845, 680), (725, 370), (145, 665), (415, 635), (510, 875), (560, 365),
        (300, 465), (520, 585), (480, 415), (835, 625), (975, 580), (1215, 245),
        (1320, 315), (1250, 400), (660, 180), (410, 250), (420, 555), (575, 665),
        (1150, 1160), (700, 580), (685, 595), (685, 610), (770, 610), (795, 645),
        (720, 635), (760, 650), (475, 960), (95, 260), (875, 920), (700, 500), (555, 815),
        (830, 485), (1170, 65), (830, 610), (605, 625), (595, 360), (1340, 725),
        (1740, 245)]),
}
OPTIMOS = {"burma14": 3323, "ulysses16": 6859, "ulysses22": 7013, "berlin52": 7542}


def matriz_distancias(coordenadas, tipo):
    # Matriz entera de distancias con las convenciones de TSPLIB.
    c = np.asarray(coordenadas, dtype=float)
    if tipo == "EUC_2D":
        dif = c[:, None, :] - c[None, :, :]
        return np.floor(np.sqrt((dif ** 2).sum(axis=-1)) + 0.5).astype(np.int64)
    if tipo == "GEO":
        grados = np.trunc(c)
        rad = 3.141592 * (grados + 5.0 * (c - grados) / 3.0) / 180.0
        lat, lon = rad[:, 0], rad[:, 1]
        q1 = np.cos(lon[:, None] - lon[None, :])
        q2 = np.cos(lat[:, None] - lat[None, :])
        q3 = np.cos(lat[:, None] + lat[None, :])
        arg = np.clip(0.5 * ((1.0 + q1) * q2 - (1.0 - q1) * q3), -1.0, 1.0)
        d = np.triu((6378.388 * np.arccos(arg) + 1.0).astype(np.int64), 1)
        return d + d.T
    raise ValueError(tipo)


def cargar_tsplib(nombre):
    tipo, coords = TSPLIB[nombre]
    return matriz_distancias(coords, tipo)


def instancia_aleatoria(n, semilla=0, lado=1000.0):
    # n puntos uniformes en [0, lado]^2 (coordenadas con un decimal).
    rng = np.random.default_rng(semilla)
    coords = np.round(rng.uniform(0.0, lado, size=(n, 2)), 1)
    return matriz_distancias(coords, "EUC_2D")


for nombre in TSPLIB:
    d = cargar_tsplib(nombre)
    print(f"{nombre:>10}: n = {d.shape[0]:>2}, simétrica = {np.array_equal(d, d.T)}")

## 2. Rutas y la vecindad 2-opt

Una ruta se representa como una permutación de $0,\dots,n-1$ leída de forma cíclica. La
forma canónica empieza en la ciudad $0$ y recorre el ciclo en el sentido en que la segunda
ciudad es menor que la última; así, las $(n-1)!/2$ rutas distintas tienen un único
representante.

El movimiento 2-opt `dos_opt(r, i, j)` invierte el tramo `r[i+1 : j+1]`: retira las aristas
$(r_i, r_{i+1})$ y $(r_j, r_{j+1})$ y agrega $(r_i, r_j)$ y $(r_{i+1}, r_{j+1})$.

In [ ]:
def costo_ruta(ruta, d):
    r = np.asarray(ruta)
    return int(d[r, np.roll(r, -1)].sum())


def normalizar_ruta(ruta):
    # Representante canónico: empieza en 0 y r[1] < r[-1].
    r = np.asarray(ruta)
    r = np.roll(r, -int(np.flatnonzero(r == 0)[0]))
    if r.size > 2 and r[-1] < r[1]:
        r = np.concatenate(([0], r[:0:-1]))
    return r


def dos_opt(ruta, i, j):
    r = np.array(ruta, copy=True)
    r[i + 1 : j + 1] = r[i + 1 : j + 1][::-1]
    return r


def enumerar_rutas(n):
    # Cada ciclo distinto una sola vez, en forma canónica: (n-1)!/2 rutas.
    for perm in itertools.permutations(range(1, n)):
        if perm[0] < perm[-1]:
            yield (0, *perm)


def fuerza_bruta(d):
    # Óptimo exacto por enumeración (solo para n pequeño).
    n = d.shape[0]
    dl = d.tolist()
    mejor, mejor_costo = None, None
    for r in enumerar_rutas(n):
        c = sum(dl[r[k]][r[k - 1]] for k in range(n))
        if mejor_costo is None or c < mejor_costo:
            mejor, mejor_costo = r, c
    return np.array(mejor), int(mejor_costo)


print("Número de rutas:", {n: sum(1 for _ in enumerar_rutas(n)) for n in range(4, 9)})

## 3. La cadena de Metropolis, el recocido simulado y el QUBO

### 3.1 Esquema de enfriamiento

El recocido usa el esquema geométrico $T_k = T_0\,\alpha^k$ con $\alpha = 0.95$, y en cada
temperatura da $L$ pasos de Metropolis: es una sucesión de cadenas homogéneas. La
temperatura inicial se elige para aceptar cerca del 80 % de las subidas típicas,
$e^{-\bar\Delta/T_0} = 0.8$, y el enfriamiento se detiene en $T_{\text{final}} = 10^{-3}\,T_0$.

In [ ]:
def temperatura_inicial(deltas, aceptacion=0.8):
    subidas = np.asarray(deltas, dtype=float)
    subidas = subidas[subidas > 0]
    if subidas.size == 0:
        return 1.0
    return float(subidas.mean() / np.log(1.0 / aceptacion))


@dataclass(frozen=True)
class Enfriamiento:
    T0: float
    T_final: float
    alfa: float = 0.95
    pasos_por_T: int = 100

    @classmethod
    def automatico(cls, deltas, pasos_por_T, aceptacion_inicial=0.8,
                   razon_final=1e-3, alfa=0.95):
        T0 = temperatura_inicial(deltas, aceptacion_inicial)
        return cls(T0=T0, T_final=razon_final * T0, alfa=alfa, pasos_por_T=pasos_por_T)

    def temperaturas(self):
        cociente = np.log(self.T_final / self.T0) / np.log(self.alfa)
        niveles = int(np.floor(cociente + 1e-12)) + 1
        return self.T0 * self.alfa ** np.arange(niveles)


@dataclass
class Resultado:
    solucion: np.ndarray
    valor: float
    historial: pd.DataFrame      # una fila por temperatura
    tiempo: float
    pasos: int
    solucion_factible: np.ndarray | None = None
    valor_factible: float | None = None

### 3.2 Metropolis y recocido sobre rutas

En cada paso se elige un par de posiciones $\{i,j\}$ uniforme entre los $\binom n2$
posibles. Los $n$ pares de aristas adyacentes son **movimientos nulos** (la ruta no cambia);
los otros $n(n-3)/2$ dan vecinos distintos, cada uno con probabilidad $1/\binom n2$, así que
la propuesta es simétrica. El cambio de costo se calcula en tiempo constante:

$$\Delta = d(x_1,y_1) + d(x_2,y_2) - d(x_1,x_2) - d(y_1,y_2),$$

y el movimiento se acepta con probabilidad $\min\{1, e^{-\Delta/T}\}$.

In [ ]:
def metropolis_rutas(ruta, costo, distancias, T, pasos, rng,
                     mejor_costo=math.inf, registro=None):
    # Da `pasos` pasos de Metropolis a temperatura fija T (modifica `ruta`).
    n = len(ruta)
    primeros = rng.integers(0, n, size=pasos).tolist()
    segundos = rng.integers(0, n - 1, size=pasos).tolist()
    uniformes = rng.random(pasos).tolist()
    aceptados = nulos = 0
    mejor_ruta = None
    for a, b, u in zip(primeros, segundos, uniformes):
        i, j = (a, b + 1) if b >= a else (b, a)        # par uniforme i < j
        if j == i + 1 or (i == 0 and j == n - 1):       # aristas adyacentes: nulo
            nulos += 1
        else:
            x1, x2 = ruta[i], ruta[i + 1]
            y1, y2 = ruta[j], ruta[(j + 1) % n]
            delta = (distancias[x1][y1] + distancias[x2][y2]
                     - distancias[x1][x2] - distancias[y1][y2])
            if delta <= 0 or u < math.exp(-delta / T):  # regla de Metropolis
                ruta[i + 1 : j + 1] = ruta[j:i:-1]
                costo += delta
                aceptados += 1
                if costo < mejor_costo:
                    mejor_costo = costo
                    mejor_ruta = ruta.copy()
        if registro is not None:
            registro.append(tuple(ruta))
    return costo, aceptados, nulos, mejor_costo, mejor_ruta


def muestra_deltas(ruta, d, rng, tamano=1000):
    # Cambios de costo de movimientos 2-opt al azar, para calibrar T0.
    n = len(ruta)
    a = rng.integers(0, n, size=tamano)
    b = rng.integers(0, n - 1, size=tamano)
    b = b + (b >= a)
    i, j = np.minimum(a, b), np.maximum(a, b)
    r = np.asarray(ruta)
    x1, x2, y1, y2 = r[i], r[i + 1], r[j], r[(j + 1) % n]
    return d[x1, y1] + d[x2, y2] - d[x1, x2] - d[y1, y2]


def recocido_rutas(d, semilla=None):
    # Recocido simulado con vecindad 2-opt y 10*C(n,2) pasos por temperatura.
    inicio = time.perf_counter()
    n = d.shape[0]
    rng = np.random.default_rng(semilla)
    r = rng.permutation(n)
    enfr = Enfriamiento.automatico(muestra_deltas(r, d, rng), pasos_por_T=5 * n * (n - 1))
    ruta, dl = r.tolist(), d.tolist()
    costo = costo_ruta(r, d)
    mejor_costo, mejor_ruta = costo, list(ruta)
    filas = []
    temperaturas = enfr.temperaturas()
    for T in temperaturas.tolist():
        costo, acept, nulos, mejor_costo, nueva = metropolis_rutas(
            ruta, costo, dl, T, enfr.pasos_por_T, rng, mejor_costo)
        if nueva is not None:
            mejor_ruta = nueva
        filas.append((T, costo, mejor_costo, acept / max(1, enfr.pasos_por_T - nulos)))
    historial = pd.DataFrame(filas, columns=["temperatura", "valor", "mejor", "aceptacion"])
    return Resultado(np.array(mejor_ruta), costo_ruta(mejor_ruta, d), historial,
                     time.perf_counter() - inicio, temperaturas.size * enfr.pasos_por_T)

### 3.3 El TSP como QUBO y el recocido con volteos de un bit

Se fija la ciudad $0$ en la posición $0$ y se usan $(n-1)^2$ variables $y_{ki}=1$ si la
ciudad $i$ ocupa la posición $k$. La energía es

$$E_M(y) = f(y) + M\,\Phi(y), \qquad
\Phi(y) = \sum_i \Big(\sum_k y_{ki} - 1\Big)^2 + \sum_k \Big(\sum_i y_{ki} - 1\Big)^2,$$

donde $f$ es el costo del recorrido. Se guarda como $y^\top Q y + \text{constante}$ con $Q$
triangular superior. En rutas factibles $E_M = f$, y en infactibles $E_M \ge M$, así que
basta $M > f^\ast$; se usa $M = U + 1$ con $U$ el costo de la ruta del vecino más cercano.

El recocido sobre el QUBO propone voltear un bit uniforme y usa el *campo local*
$h_k + (Ay)_k$ para calcular $\Delta E$ en tiempo constante.

In [ ]:
@dataclass
class QUBO:
    Q: np.ndarray            # triangular superior; la diagonal son términos lineales
    constante: float = 0.0

    def __post_init__(self):
        Q = np.asarray(self.Q, dtype=float)
        self.Q = np.triu(Q) + np.triu(Q.T, 1)

    @property
    def n_variables(self):
        return self.Q.shape[0]

    def energia(self, x):
        X = np.asarray(x, dtype=float)
        if X.ndim == 1:
            return float(X @ self.Q @ X + self.constante)
        return ((X @ self.Q) * X).sum(axis=1) + self.constante


def ruta_vecino_mas_cercano(d):
    n = d.shape[0]
    ruta, libres = [0], set(range(1, n))
    while libres:
        siguiente = min(libres, key=lambda j: d[ruta[-1], j])
        ruta.append(siguiente)
        libres.remove(siguiente)
    return np.array(ruta)


def penalizacion_segura(d):
    # M = U + 1, con U >= z* el costo de la ruta del vecino más cercano.
    return costo_ruta(ruta_vecino_mas_cercano(d), d) + 1.0


def construir_qubo(d, M=None):
    d = np.asarray(d, dtype=float)
    n = d.shape[0]
    if M is None:
        M = penalizacion_segura(d)
    m = n - 1
    Q = np.zeros((m * m, m * m))
    D = d[1:, 1:]
    for k in range(m - 1):                       # tramo entre posiciones k+1 y k+2
        Q[k * m:(k + 1) * m, (k + 1) * m:(k + 2) * m] += D
    lineal = np.zeros(m * m)
    lineal[:m] += d[0, 1:]                       # de la ciudad 0 a la posición 1
    lineal[(m - 1) * m:] += d[1:, 0]             # de la última posición a la ciudad 0
    posicion = np.repeat(np.arange(m), m)
    ciudad = np.tile(np.arange(m), m)
    comparten = (posicion[:, None] == posicion[None, :]) | (ciudad[:, None] == ciudad[None, :])
    Q += 2 * M * np.triu(comparten, 1)           # penalización: términos cruzados
    lineal -= 2 * M                              # penalización: términos lineales
    Q += np.diag(lineal)
    return QUBO(Q, constante=2 * m * M)


def penalizacion(x):
    # Phi(y): cero exactamente en las asignaciones factibles.
    X = np.asarray(x).astype(int)
    m = math.isqrt(X.shape[-1])
    Y = X.reshape(-1, m, m)
    p = ((1 - Y.sum(axis=1)) ** 2).sum(axis=1) + ((1 - Y.sum(axis=2)) ** 2).sum(axis=1)
    return int(p[0]) if X.ndim == 1 else p


def decodificar(x):
    v = np.asarray(x).astype(int)
    m = math.isqrt(v.size)
    Y = v.reshape(m, m)
    if np.all(Y.sum(axis=0) == 1) and np.all(Y.sum(axis=1) == 1):
        return np.concatenate(([0], Y.argmax(axis=1) + 1))
    return None


def umbral_exacto(d):
    # Menor M a partir del cual todo mínimo de E_M es factible (enumera 2^((n-1)^2)).
    _, z = fuerza_bruta(d)
    costo = construir_qubo(d, M=0.0)
    m = costo.n_variables
    bits = np.arange(m)
    umbral = 0.0
    for inicio in range(0, 1 << m, 1 << 16):
        ids = np.arange(inicio, min(inicio + (1 << 16), 1 << m))
        X = ((ids[:, None] >> bits) & 1).astype(np.int8)
        p = penalizacion(X)
        inf = p > 0
        if inf.any():
            umbral = max(umbral, float(((z - costo.energia(X[inf])) / p[inf]).max()))
    return umbral


def recocido_qubo(qubo, semilla=None, es_factible=None):
    # Recocido con volteos de un bit y 10*m pasos por temperatura.
    inicio = time.perf_counter()
    rng = np.random.default_rng(semilla)
    m = qubo.n_variables
    h = np.diag(qubo.Q).copy()
    A = qubo.Q + qubo.Q.T
    np.fill_diagonal(A, 0.0)
    x = rng.integers(0, 2, size=m).astype(np.int8)
    campo = A @ x
    enfr = Enfriamiento.automatico((1 - 2 * x) * (h + campo), pasos_por_T=10 * m)
    E = qubo.energia(x)
    mejor_E, mejor_x = E, x.copy()
    fact_E, fact_x = math.inf, None
    if es_factible is not None and es_factible(x):
        fact_E, fact_x = E, x.copy()
    hl = h.tolist()
    filas = []
    temperaturas = enfr.temperaturas()
    for T in temperaturas.tolist():
        ks = rng.integers(0, m, size=enfr.pasos_por_T).tolist()
        us = rng.random(enfr.pasos_por_T).tolist()
        aceptados = 0
        for k, u in zip(ks, us):
            s = 1 - 2 * int(x[k])
            delta = s * (hl[k] + campo[k])               # cambio de energía en O(1)
            if delta <= 0 or u < math.exp(-delta / T):
                x[k] = 1 - x[k]
                campo += s * A[k]
                E += delta
                aceptados += 1
                if E < mejor_E - 1e-9:
                    mejor_E, mejor_x = E, x.copy()
                if es_factible is not None and E < fact_E - 1e-9 and es_factible(x):
                    fact_E, fact_x = E, x.copy()
        filas.append((T, E, mejor_E, aceptados / enfr.pasos_por_T,
                      None if es_factible is None else bool(es_factible(x))))
    historial = pd.DataFrame(filas, columns=["temperatura", "valor", "mejor",
                                             "aceptacion", "factible"])
    return Resultado(mejor_x, float(qubo.energia(mejor_x)), historial,
                     time.perf_counter() - inicio, temperaturas.size * enfr.pasos_por_T,
                     fact_x, None if fact_x is None else float(qubo.energia(fact_x)))

### 3.4 La cadena exacta para $n$ pequeño

Con $n=5$ hay $12$ rutas, así que se puede construir la matriz de transición $P_T$ completa
con la misma propuesta del recocido, calcular su distribución estacionaria y su espectro, y
medir la profundidad de Hajek de cada mínimo local.

In [ ]:
@dataclass
class Cadena:
    estados: list
    costos: np.ndarray
    vecinos: np.ndarray      # vecinos[a, b]: b != a se obtiene de a con un 2-opt
    P: np.ndarray
    T: float


def cadena_exacta(d, T):
    n = d.shape[0]
    estados = list(enumerar_rutas(n))
    indice = {s: a for a, s in enumerate(estados)}
    costos = np.array([costo_ruta(s, d) for s in estados], dtype=float)
    pares = [(i, j) for i in range(n - 1) for j in range(i + 1, n)]
    N = len(estados)
    P = np.zeros((N, N))
    vecinos = np.zeros((N, N), dtype=bool)
    for a, s in enumerate(estados):
        r = np.array(s)
        for i, j in pares:
            b = indice[tuple(normalizar_ruta(dos_opt(r, i, j)).tolist())]
            if b != a:
                vecinos[a, b] = True
            P[a, b] += math.exp(-max(costos[b] - costos[a], 0.0) / T)
        P[a] /= len(pares)
        P[a, a] += 1.0 - P[a].sum()
    return Cadena(estados, costos, vecinos, P, float(T))


def boltzmann(costos, T):
    c = np.asarray(costos, dtype=float)
    w = np.exp(-(c - c.min()) / T)
    return w / w.sum()


def estacionaria(P):
    # Resuelve pi P = pi con sum(pi) = 1.
    N = P.shape[0]
    A = P.T - np.eye(N)
    A[-1] = 1.0
    b = np.zeros(N)
    b[-1] = 1.0
    return np.linalg.solve(A, b)


def error_balance(P, pi):
    F = pi[:, None] * P
    return float(np.abs(F - F.T).max())


def autovalores(cadena):
    # Autovalores de P_T a partir de la matriz simétrica D^{1/2} P D^{-1/2}.
    c = cadena.costos
    razon = np.exp((c[None, :] - c[:, None]) / (2 * cadena.T))
    S = np.where(cadena.vecinos, cadena.P * razon, 0.0)
    np.fill_diagonal(S, np.diag(cadena.P))
    return np.linalg.eigvalsh((S + S.T) / 2)


def brecha_espectral(cadena):
    # Brecha absoluta 1 - max(|lambda_2|, |lambda_N|).
    ev = autovalores(cadena)
    return float(1.0 - max(abs(ev[0]), abs(ev[-2])))


def distancia_tv(p, q):
    return float(0.5 * np.abs(np.asarray(p) - np.asarray(q)).sum())


def minimos_locales(cadena):
    c = cadena.costos
    return [a for a in range(c.size) if np.all(c[cadena.vecinos[a]] >= c[a])]


def profundidad(cadena, a):
    # Menor subida h tal que desde a se llega a un estado mejor sin superar f(a) + h.
    c = cadena.costos
    for nivel in np.unique(c[c >= c[a]]):
        permitidos = c <= nivel
        visitados, pila = {a}, [a]
        while pila:
            u = pila.pop()
            for v in np.flatnonzero(cadena.vecinos[u] & permitidos).tolist():
                if v not in visitados:
                    visitados.add(v)
                    pila.append(v)
        if c[list(visitados)].min() < c[a]:
            return float(nivel - c[a])
    return math.inf


def profundidad_critica(cadena):
    # h* (d* de Hajek): máxima profundidad entre los mínimos locales no globales.
    c = cadena.costos
    espurios = [a for a in minimos_locales(cadena) if c[a] > c.min()]
    return max((profundidad(cadena, a) for a in espurios), default=0.0)


def trayectoria_rutas(d, T, pasos, semilla=None):
    # Simula la cadena homogénea con el núcleo del recocido; devuelve índices de estado.
    estados = list(enumerar_rutas(d.shape[0]))
    indice = {s: a for a, s in enumerate(estados)}
    rng = np.random.default_rng(semilla)
    ruta = list(estados[0])
    registro = []
    metropolis_rutas(ruta, costo_ruta(ruta, d), d.tolist(), T, pasos, rng, registro=registro)
    canonico = {t: indice[tuple(normalizar_ruta(np.array(t)).tolist())] for t in set(registro)}
    return np.array([canonico[t] for t in registro])

## 4. Verificación de las propiedades de la cadena

Antes de los experimentos se comprueban numéricamente las proposiciones de la sección
*Marco teórico* para $n = 4,\dots,7$: número de vecinos $n(n-3)/2$, simetría de la vecindad,
irreducibilidad (con el diámetro del grafo de vecindad, que debe ser a lo sumo $n-2$),
si el grafo es bipartito, balance detallado respecto de Boltzmann y que, con $n=5$, ningún
valor propio de $P_T$ es negativo.

In [ ]:
def diametro(vecinos):
    N = vecinos.shape[0]
    A = vecinos.astype(int)
    alcanzado, k = np.eye(N, dtype=int), 0
    while not alcanzado.all():
        alcanzado = ((alcanzado + alcanzado @ A) > 0).astype(int)
        k += 1
        if k > N:
            return math.inf
    return k


filas = []
for n in range(4, 8):
    d = instancia_aleatoria(n, semilla=0)
    c = cadena_exacta(d, T=100.0)
    grados = set(c.vecinos.sum(axis=1).tolist())
    ev_vecindad = np.linalg.eigvalsh(c.vecinos.astype(float))
    ev = autovalores(c)
    filas.append({
        "n": n,
        "rutas": len(c.estados),
        "vecinos por ruta": sorted(grados),
        "n(n-3)/2": n * (n - 3) // 2,
        "vecindad simétrica": bool((c.vecinos == c.vecinos.T).all()),
        "diámetro": diametro(c.vecinos),
        "n-2": n - 2,
        "bipartito": bool(np.isclose(ev_vecindad.min(), -ev_vecindad.max())),
        "error balance": error_balance(c.P, boltzmann(c.costos, c.T)),
        "min autovalor P_T": ev.min(),
        "2δ-1": 2 * (2 / (n - 1)) - 1,
    })
verificacion = pd.DataFrame(filas)
verificacion.to_csv(RESULTADOS / "verificacion_cadena.csv", index=False)
verificacion

## 5. Experimento 1: la distribución de Boltzmann según la temperatura

Con la instancia aleatoria de $n=5$ ciudades (semilla 0) se calcula $\pi_T$ exacta para
varias temperaturas, expresadas como múltiplos del rango de costos
$\max f - \min f$. A temperatura alta $\pi_T$ es casi uniforme; a temperatura baja se
concentra en la ruta óptima.

In [ ]:
d5 = instancia_aleatoria(5, semilla=0)
rango5 = float(np.ptp(cadena_exacta(d5, 1.0).costos))
c1 = cadena_exacta(d5, 1.0)
orden = np.argsort(c1.costos, kind="stable")

factores_figura = [2.0, 0.5, 0.1]
colores = [AMARILLO, NARANJA, AZUL]
fig, ax = plt.subplots(figsize=(7, 3.2))
ancho = 0.27
x = np.arange(len(orden))
for k, (fac, col) in enumerate(zip(factores_figura, colores)):
    pi = boltzmann(c1.costos, fac * rango5)[orden]
    ax.bar(x + (k - 1) * ancho, pi, width=ancho - 0.03, color=col,
           label=f"$T = {fac:g}\\,(\\max f-\\min f)$")
ax.axhline(1 / 12, color=TINTA_2, lw=1, ls="--")
ax.text(len(orden) - 0.5, 1 / 12 + 0.01, "uniforme (1/12)", ha="right", color=TINTA_2, fontsize=8)
ax.set_xticks(x, [f"{int(c)}" for c in c1.costos[orden]], rotation=45)
ax.set_xlabel("costo de la ruta (rutas ordenadas de menor a mayor costo)")
ax.set_ylabel("$\\pi_T(x)$")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper right")
guardar(fig, "exp1_boltzmann_temperatura")
plt.show()

# Costo medio bajo pi_T: no decreciente en T (derivada = Var/T^2).
Ts = np.geomspace(0.01, 10, 60) * rango5
media = [float((boltzmann(c1.costos, T) * c1.costos).sum()) for T in Ts]
print("¿E[f] no decreciente en T?", bool(np.all(np.diff(media) >= -1e-9)))
print(f"rango de costos = {rango5:.0f}, costo óptimo = {c1.costos.min():.0f}")

## 6. Experimento 2: frecuencias empíricas frente a la distribución exacta

A temperatura fija $T = 0.5\,(\max f - \min f)$ se simula una trayectoria de $200\,000$
pasos con el mismo núcleo del recocido (semilla 1) y se comparan las frecuencias de visita
con $\pi_T$ exacta. Por el teorema ergódico, la distancia de variación total entre ambas
debe tender a cero.

In [ ]:
PASOS = 200_000
T_sim = 0.5 * rango5
c_sim = cadena_exacta(d5, T_sim)
pi_sim = boltzmann(c_sim.costos, T_sim)
pi_num = estacionaria(c_sim.P)
print("max |pi(numérica) - Boltzmann| =", np.abs(pi_num - pi_sim).max())

tray = trayectoria_rutas(d5, T_sim, PASOS, semilla=1)
k = len(c_sim.estados)
empirica = np.bincount(tray, minlength=k) / tray.size
orden = np.argsort(c_sim.costos, kind="stable")
frecuencias = pd.DataFrame({
    "ruta": ["-".join(map(str, c_sim.estados[a])) for a in orden],
    "costo": c_sim.costos[orden].astype(int),
    "pi_exacta": pi_sim[orden],
    "empirica": empirica[orden],
})
frecuencias.to_csv(RESULTADOS / "exp2_frecuencias.csv", index=False)

puntos = np.unique(np.geomspace(100, PASOS, 30).astype(int))
tv = [distancia_tv(np.bincount(tray[:t], minlength=k) / t, pi_sim) for t in puntos]
convergencia = pd.DataFrame({"pasos": puntos, "tv": tv})
convergencia.to_csv(RESULTADOS / "exp2_convergencia.csv", index=False)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={"width_ratios": [1.4, 1]})
x = np.arange(k)
a1.bar(x - 0.2, frecuencias["pi_exacta"], width=0.37, color=AZUL, label="$\\pi_T$ exacta")
a1.bar(x + 0.2, frecuencias["empirica"], width=0.37, color=NARANJA, label="frecuencia empírica")
a1.set_xticks(x, frecuencias["costo"], rotation=45)
a1.set_xlabel("costo de la ruta")
a1.set_ylabel("probabilidad")
a1.grid(axis="x", visible=False)
a1.legend()
a2.loglog(convergencia["pasos"], convergencia["tv"], color=AZUL, marker="o", ms=4)
ref = convergencia["tv"].iloc[0] * np.sqrt(convergencia["pasos"].iloc[0] / convergencia["pasos"])
a2.loglog(convergencia["pasos"], ref, color=TINTA_2, lw=1, ls="--", label="pendiente $-1/2$")
a2.set_xlabel("pasos $t$")
a2.set_ylabel("$\\|\\hat\\pi_t - \\pi_T\\|_{TV}$")
a2.legend()
fig.tight_layout()
guardar(fig, "exp2_frecuencias_convergencia")
plt.show()

print(f"TV final con {PASOS} pasos: {tv[-1]:.4f}")
frecuencias

## 7. Experimento 3: brecha espectral y profundidad crítica

Para la instancia de la semilla 0 y para una instancia de contraste con un único mínimo
local (la primera semilla que lo cumple) se calcula la brecha espectral $\gamma(T)$ en un
rango de temperaturas. Según el teorema de la brecha a temperatura baja,
$-\log\gamma(T) \approx h^\ast/T$, así que la pendiente de $-\log\gamma$ frente a $1/T$ en
las dos temperaturas más bajas debe aproximarse a la profundidad crítica $h^\ast$,
calculada de forma exacta. En la instancia sin mínimos espurios, $h^\ast = 0$ y la brecha
no se anula.

In [ ]:
FACTORES_T = [5.0, 2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005]


def buscar_semilla(minimos, max_semillas=500):
    for s in range(max_semillas):
        if len(minimos_locales(cadena_exacta(instancia_aleatoria(5, semilla=s), 1.0))) == minimos:
            return s
    return None


def tabla_temperaturas(semilla):
    d = instancia_aleatoria(5, semilla=semilla)
    rango = float(np.ptp(cadena_exacta(d, 1.0).costos))
    filas = []
    for fac in FACTORES_T:
        c = cadena_exacta(d, fac * rango)
        pi = estacionaria(c.P)
        b = brecha_espectral(c)
        filas.append({
            "semilla": semilla,
            "minimos_locales": len(minimos_locales(c)),
            "h_critica": profundidad_critica(c),
            "T/rango": fac,
            "T": c.T,
            "brecha": b,
            "t_relajacion": 1 / b,
            "pi_optimo": pi[c.costos == c.costos.min()].sum(),
            "error_boltzmann": np.abs(pi - boltzmann(c.costos, c.T)).max(),
            "error_balance": error_balance(c.P, pi),
        })
    return pd.DataFrame(filas)


contraste = buscar_semilla(minimos=1)
temperaturas = pd.concat([tabla_temperaturas(s) for s in [0, contraste]], ignore_index=True)
temperaturas.to_csv(RESULTADOS / "exp3_temperaturas.csv", index=False)

pendientes = {}
for s, g in temperaturas.groupby("semilla"):
    bajo = g.nsmallest(2, "T")
    pendientes[s] = float((-np.diff(np.log(bajo["brecha"])) / np.diff(1 / bajo["T"]))[0])
    print(f"semilla {s}: mínimos locales = {g['minimos_locales'].iloc[0]}, "
          f"h* exacta = {g['h_critica'].iloc[0]:.1f}, "
          f"pendiente de -ln(brecha) frente a 1/T = {pendientes[s]:.1f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
for (s, g), col in zip(temperaturas.groupby("semilla"), [AZUL, NARANJA]):
    etiqueta = f"semilla {s} ($h^*={g['h_critica'].iloc[0]:.0f}$)"
    a1.loglog(g["T/rango"], g["brecha"], color=col, marker="o", ms=4, label=etiqueta)
    a2.plot(1 / g["T"], -np.log(g["brecha"]), color=col, marker="o", ms=4, label=etiqueta)
g0 = temperaturas[temperaturas["semilla"] == 0]
x0 = 1 / g0["T"].to_numpy()
h0 = g0["h_critica"].iloc[0]
a2.plot(x0, -np.log(g0["brecha"].iloc[-1]) + h0 * (x0 - x0[-1]), color=TINTA_2, lw=1, ls="--",
        label=f"pendiente $h^*={h0:.0f}$")
a1.set_xlabel("$T/(\\max f-\\min f)$")
a1.set_ylabel("brecha espectral $\\gamma(T)$")
a1.legend()
a2.set_xlabel("$1/T$")
a2.set_ylabel("$-\\ln\\gamma(T)$")
a2.legend()
fig.tight_layout()
guardar(fig, "exp3_brecha_espectral")
plt.show()

temperaturas

## 8. Experimento 4: recocido simulado en instancias de TSPLIB

Se corre el recocido sobre rutas en `burma14`, `ulysses16`, `ulysses22` y `berlin52`, con
10 semillas por instancia (40 corridas), y se compara el mejor costo encontrado con el
óptimo publicado. Para `berlin52` se grafica la evolución por nivel de temperatura de la
semilla 0.

In [ ]:
INSTANCIAS = ["burma14", "ulysses16", "ulysses22", "berlin52"]
SEMILLAS = range(10)

filas, curvas = [], {}
for nombre in INSTANCIAS:
    d = cargar_tsplib(nombre)
    for s in SEMILLAS:
        res = recocido_rutas(d, semilla=s)
        filas.append({"instancia": nombre, "n": d.shape[0], "semilla": s,
                      "costo": res.valor, "optimo": OPTIMOS[nombre],
                      "brecha_%": 100 * (res.valor - OPTIMOS[nombre]) / OPTIMOS[nombre],
                      "pasos": res.pasos, "tiempo_s": round(res.tiempo, 2)})
        if s == 0:
            curvas[nombre] = res.historial
    print(f"{nombre:>10}: listo")

corridas_tsplib = pd.DataFrame(filas)
corridas_tsplib.to_csv(RESULTADOS / "exp4_corridas.csv", index=False)
resumen_tsplib = (corridas_tsplib.groupby(["instancia", "n", "optimo"], sort=False)
                  .agg(brecha_media=("brecha_%", "mean"), brecha_max=("brecha_%", "max"),
                       veces_optimo=("brecha_%", lambda b: int((b == 0).sum())),
                       pasos=("pasos", "first"), tiempo_medio_s=("tiempo_s", "mean"))
                  .round(2).reset_index())
resumen_tsplib.to_csv(RESULTADOS / "exp4_resumen.csv", index=False)

h = curvas["berlin52"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
a1.plot(h.index, h["valor"], color=AZUL, label="costo actual")
a1.plot(h.index, h["mejor"], color=NARANJA, label="mejor costo")
a1.axhline(OPTIMOS["berlin52"], color=TINTA_2, lw=1, ls="--", label="óptimo (7542)")
a1.set_yscale("log")
a1.set_xlabel("nivel de temperatura $k$")
a1.set_ylabel("costo")
a1.legend()
a2.plot(h.index, h["aceptacion"], color=AZUL)
a2.set_xlabel("nivel de temperatura $k$")
a2.set_ylabel("tasa de aceptación")
fig.tight_layout()
guardar(fig, "exp4_berlin52")
plt.show()

resumen_tsplib

## 9. Experimento 5: recocido sobre el QUBO y sensibilidad a $M$

Para tres instancias (aleatorias con $n=5$ y $n=8$, y `burma14`) se corre el recocido con
volteos de un bit para varios valores de $M$: múltiplos de $\max d$, la cota segura
$U + 1$ y, cuando se puede calcular, $1.1\,M^\ast$ con $M^\ast$ el umbral exacto. Se
registra si el estado final es una ruta, si la cadena visitó alguna ruta y la brecha de la
mejor ruta visitada. Como referencia, se corre el recocido sobre rutas con las mismas
semillas.

In [ ]:
FACTORES_M = [0.25, 0.5, 1.0, 2.0, 4.0]


def es_ruta(x):
    return penalizacion(x) == 0


casos = []
d5 = instancia_aleatoria(5, semilla=0)
casos.append(("aleatoria_n5_s0", d5, fuerza_bruta(d5)[1], umbral_exacto(d5)))
d8 = instancia_aleatoria(8, semilla=0)
casos.append(("aleatoria_n8_s0", d8, fuerza_bruta(d8)[1], None))
casos.append(("burma14", cargar_tsplib("burma14"), OPTIMOS["burma14"], None))

filas = []
for nombre, d, optimo, umbral in casos:
    max_d = float(d.max())
    valores = {f"{f:g}·max d": f * max_d for f in FACTORES_M}
    valores["seguro (U+1)"] = penalizacion_segura(d)
    if umbral is not None:
        valores["1.1·M*"] = 1.1 * umbral
    for etiqueta, M in valores.items():
        q = construir_qubo(d, M)
        for s in SEMILLAS:
            res = recocido_qubo(q, semilla=s, es_factible=es_ruta)
            visito = res.solucion_factible is not None
            filas.append({"instancia": nombre, "metodo": "QUBO", "M": etiqueta,
                          "valor_M": round(M, 1), "M/max d": M / max_d, "semilla": s,
                          "final_es_ruta": decodificar(res.solucion) is not None,
                          "visito_ruta": visito,
                          "costo": res.valor_factible if visito else np.nan,
                          "optimo": optimo, "pasos": res.pasos, "tiempo_s": res.tiempo})
        print(f"{nombre:>16}  M = {etiqueta:<13} listo")
    for s in SEMILLAS:
        res = recocido_rutas(d, semilla=s)
        filas.append({"instancia": nombre, "metodo": "rutas (2-opt)", "M": "—",
                      "valor_M": np.nan, "M/max d": np.nan, "semilla": s,
                      "final_es_ruta": True, "visito_ruta": True, "costo": res.valor,
                      "optimo": optimo, "pasos": res.pasos, "tiempo_s": res.tiempo})

corridas_qubo = pd.DataFrame(filas)
corridas_qubo["brecha_%"] = 100 * (corridas_qubo["costo"] - corridas_qubo["optimo"]) / corridas_qubo["optimo"]
corridas_qubo.to_csv(RESULTADOS / "exp5_corridas.csv", index=False)
resumen_qubo = (corridas_qubo.groupby(["instancia", "metodo", "M"], sort=False)
                .agg(valor_M=("valor_M", "first"), M_sobre_max_d=("M/max d", "first"),
                     final_es_ruta_pct=("final_es_ruta", "mean"),
                     visito_ruta_pct=("visito_ruta", "mean"),
                     brecha_media=("brecha_%", "mean"),
                     veces_optimo=("brecha_%", lambda b: int((b.abs() < 1e-9).sum())),
                     pasos=("pasos", "first"))
                .reset_index())
resumen_qubo[["final_es_ruta_pct", "visito_ruta_pct"]] *= 100
resumen_qubo = resumen_qubo.round(2)
resumen_qubo.to_csv(RESULTADOS / "exp5_resumen.csv", index=False)
if umbral_n5 := casos[0][3]:
    print(f"umbral exacto M* (n = 5, semilla 0) = {umbral_n5:.2f}  "
          f"(M*/max d = {umbral_n5 / float(d5.max()):.3f})")

qubo = resumen_qubo[resumen_qubo["metodo"] == "QUBO"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
for (nombre, g), col in zip(qubo.groupby("instancia", sort=False), [AZUL, NARANJA, AGUAMARINA]):
    g = g.sort_values("M_sobre_max_d")
    a1.plot(g["M_sobre_max_d"], g["brecha_media"], color=col, marker="o", ms=5, label=nombre)
    a2.plot(g["M_sobre_max_d"], g["final_es_ruta_pct"], color=col, marker="o", ms=5, label=nombre)
a1.set_xscale("log")
for ax in (a1, a2):
    ax.set_xticks([0.25, 0.5, 1, 2, 4], ["0.25", "0.5", "1", "2", "4"])
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
a1.set_xlabel("$M/\\max d$")
a1.set_ylabel("brecha media de la mejor ruta visitada (%)")
a1.legend()
a2.set_xlabel("$M/\\max d$")
a2.set_ylabel("corridas que terminan en una ruta (%)")
a2.set_ylim(-5, 105)
fig.tight_layout()
guardar(fig, "exp5_qubo_sensibilidad_m")
plt.show()

resumen_qubo

## 10. Descarga de resultados

En Colab, la celda siguiente comprime la carpeta `results/` y la descarga, para copiar las
figuras y tablas al repositorio (`proyecto-final-recocido-tsp/results/`). Fuera de Colab no
hace nada.

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("results", "zip", RESULTADOS)
    files.download("results.zip")
except ImportError:
    print("Fuera de Colab: los resultados ya están en", RESULTADOS.resolve())

sorted(p.name for p in RESULTADOS.iterdir())